# YOLO11n-seg + SimAM+CA + Label Smoothing + Calibration (Nhom D - D2)

Patches classification BCE targets with smoothing=0.05, then selects a predict confidence threshold on validation to control healthy false positives.

Base architecture: YOLO11n-seg + CoordAtt->SimAM on P3/P4/P5 head outputs.


In [1]:
# Install dependencies
import importlib.util
import subprocess
import sys


def ensure_package(import_name, pip_name=None):
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name or import_name])


ensure_package("roboflow")
ensure_package("ultralytics", "ultralytics==8.4.61")
ensure_package("yaml", "pyyaml")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.0/250.0 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 64.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 109.4 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 113.6 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you hav

In [2]:
# GPU check
import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))


torch: 2.10.0+cu128
cuda available: True
device: Tesla T4


In [3]:
# Download dataset from Roboflow
import os

from roboflow import Roboflow

ROBOFLOW_API_KEY_DIRECT = ""  # Optional. Prefer Kaggle secret or ROBOFLOW_API_KEY env var.
ROBOFLOW_WORKSPACE = "lets-try-this"
ROBOFLOW_PROJECT = "shrimpdishandsegv2"
ROBOFLOW_VERSION = 1
ROBOFLOW_FORMAT = "yolo26"


def get_roboflow_api_key():
    if ROBOFLOW_API_KEY_DIRECT.strip():
        return ROBOFLOW_API_KEY_DIRECT.strip()
    try:
        from kaggle_secrets import UserSecretsClient

        key = UserSecretsClient().get_secret("ROBOFLOW_API_KEY")
        if key:
            return key
    except Exception:
        pass
    return os.environ.get("ROBOFLOW_API_KEY", "").strip()


api_key = get_roboflow_api_key()
if not api_key:
    raise RuntimeError("Missing Roboflow API key. Set Kaggle Secret ROBOFLOW_API_KEY or env var ROBOFLOW_API_KEY.")

rf = Roboflow(api_key=api_key)
project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
version = project.version(ROBOFLOW_VERSION)
dataset = version.download(ROBOFLOW_FORMAT)


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to shrimpDisHandSegV2-1 in yolo26:: 100%|██████████| 2301/2301 [00:00<00:00, 3015.98it/s]


In [4]:
# Grouped-stratified split by shrimp_id to prevent leakage
import os
import random
import re
import shutil
from collections import Counter, defaultdict
from pathlib import Path

SEED = 42
random.seed(SEED)

base_path = "/kaggle/working/shrimpDisHandSegV2-1"
train_path = os.path.join(base_path, "train")
IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

GROUP_SPLIT_BY_SHRIMP = True
GROUP_STRATIFY_BY_DISEASE = True
REBUILD_SPLIT_FROM_ALL_SPLITS = True
TRAIN_RATIO = 0.80
VAL_RATIO = 0.10

SHRIMP_NAME_PATTERN = re.compile(
    r"^(?P<disease>Healthy|BG|WSSV_BG|WSSV)-(?P<shrimp_id>.+)-img-(?P<img_num>\d+)$",
    re.IGNORECASE,
)


def normalize_roboflow_stem(stem):
    stem = re.sub(r"_(jpg|jpeg|png|bmp|webp)\.rf\.[0-9a-f]+$", "", stem, flags=re.IGNORECASE)
    stem = re.sub(r"\.rf\.[0-9a-f]+$", "", stem, flags=re.IGNORECASE)
    return stem


for split in ["train", "valid", "test"]:
    for sub in ["images", "labels"]:
        os.makedirs(os.path.join(base_path, split, sub), exist_ok=True)


def parse_shrimp_group_key(image_name):
    stem = normalize_roboflow_stem(Path(image_name).stem)
    match = SHRIMP_NAME_PATTERN.match(stem)
    if not match:
        return f"unparsed::{Path(image_name).stem}", "unparsed", None, None
    disease = match.group("disease")
    shrimp_id = match.group("shrimp_id")
    img_num = int(match.group("img_num"))
    return f"{disease.lower()}::{shrimp_id}", disease, shrimp_id, img_num


def image_files_in_split(split):
    image_dir = Path(base_path) / split / "images"
    return sorted(p for p in image_dir.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)


def move_image_and_label(image_path, target_split):
    target_img_dir = Path(base_path) / target_split / "images"
    target_lbl_dir = Path(base_path) / target_split / "labels"
    target_img_dir.mkdir(parents=True, exist_ok=True)
    target_lbl_dir.mkdir(parents=True, exist_ok=True)

    label_name = f"{image_path.stem}.txt"
    label_src = image_path.parent.parent / "labels" / label_name
    image_dst = target_img_dir / image_path.name
    label_dst = target_lbl_dir / label_name

    if image_path.resolve() != image_dst.resolve():
        if image_dst.exists():
            raise FileExistsError(f"Duplicate image destination: {image_dst}")
        shutil.move(str(image_path), str(image_dst))

    if label_src.exists():
        if label_src.resolve() != label_dst.resolve():
            if label_dst.exists():
                raise FileExistsError(f"Duplicate label destination: {label_dst}")
            shutil.move(str(label_src), str(label_dst))
    else:
        label_dst.write_text("")


def rebuild_train_pool_from_all_splits():
    all_images = []
    for split in ["train", "valid", "test"]:
        all_images.extend(image_files_in_split(split))
    for image_path in sorted(all_images):
        move_image_and_label(image_path, "train")
    return image_files_in_split("train")


def remove_yolo_label_caches(root):
    for cache_path in Path(root).glob("**/*.cache"):
        cache_path.unlink()
        print(f"Removed stale cache: {cache_path}")


def disease_for_group(filenames):
    diseases = []
    for filename in filenames:
        _, disease, _, _ = parse_shrimp_group_key(filename)
        diseases.append(disease)
    counts = Counter(diseases)
    if len(counts) > 1:
        print(f"Warning: group has mixed disease names: {dict(counts)}")
    return counts.most_common(1)[0][0]


def split_one_stratum(items):
    n = len(items)
    train_count = int(TRAIN_RATIO * n)
    val_count = int(VAL_RATIO * n)
    test_count = n - train_count - val_count
    if n >= 3:
        if val_count == 0:
            val_count = 1
            train_count -= 1
        if test_count == 0:
            test_count = 1
            train_count -= 1
    if train_count < 1 and n > 0:
        train_count = 1
    while train_count + val_count + test_count > n:
        train_count -= 1
    test_count = n - train_count - val_count
    return items[:train_count], items[train_count : train_count + val_count], items[train_count + val_count :]


def grouped_stratified_split(group_items):
    strata = defaultdict(list)
    for group_key, filenames in group_items:
        strata[disease_for_group(filenames)].append((group_key, filenames))

    split_to_groups = {"train": [], "valid": [], "test": []}
    rng = random.Random(SEED)
    for disease, items in sorted(strata.items()):
        items = sorted(items, key=lambda item: item[0])
        rng.shuffle(items)
        train_items, val_items, test_items = split_one_stratum(items)
        split_to_groups["train"].extend(train_items)
        split_to_groups["valid"].extend(val_items)
        split_to_groups["test"].extend(test_items)
        print(f"  - {disease}: {len(train_items)} train groups, {len(val_items)} valid groups, {len(test_items)} test groups")
    return {split: sorted(groups, key=lambda item: item[0]) for split, groups in split_to_groups.items()}


def split_summary(split_groups):
    group_diseases = Counter()
    image_diseases = Counter()
    for _, filenames in split_groups:
        group_diseases[disease_for_group(filenames)] += 1
        for filename in filenames:
            _, disease, _, _ = parse_shrimp_group_key(filename)
            image_diseases[disease] += 1
    return group_diseases, image_diseases


def split_grouped_by_shrimp():
    image_paths = rebuild_train_pool_from_all_splits() if REBUILD_SPLIT_FROM_ALL_SPLITS else image_files_in_split("train")
    groups = defaultdict(list)
    disease_counts = Counter()
    unparsed = []

    for image_path in image_paths:
        group_key, disease, shrimp_id, img_num = parse_shrimp_group_key(image_path.name)
        groups[group_key].append(image_path.name)
        disease_counts[disease] += 1
        if disease == "unparsed":
            unparsed.append(image_path.name)

    print("Building shrimp-grouped, disease-stratified split:")
    split_to_groups = grouped_stratified_split(sorted(groups.items(), key=lambda item: item[0]))

    for split, split_groups in split_to_groups.items():
        for _, filenames in split_groups:
            for filename in filenames:
                move_image_and_label(Path(base_path) / "train" / "images" / filename, split)

    print("Shrimp-grouped split complete:")
    for split, split_groups in split_to_groups.items():
        image_count = sum(len(filenames) for _, filenames in split_groups)
        group_diseases, image_diseases = split_summary(split_groups)
        print(f"  - {split}: {len(split_groups)} shrimp groups, {image_count} images")
        print(f"    group disease counts: {dict(sorted(group_diseases.items()))}")
        print(f"    image disease counts: {dict(sorted(image_diseases.items()))}")

    print("Source filename disease counts before split:", dict(sorted(disease_counts.items())))
    if unparsed:
        print(f"Warning: {len(unparsed)} unparsed filenames. First examples: {unparsed[:10]}")

    group_to_split = {}
    leakage = []
    for split in ["train", "valid", "test"]:
        for image_path in image_files_in_split(split):
            group_key, *_ = parse_shrimp_group_key(image_path.name)
            previous_split = group_to_split.setdefault(group_key, split)
            if previous_split != split:
                leakage.append((group_key, previous_split, split, image_path.name))
    if leakage:
        raise RuntimeError(f"Shrimp-level split leakage detected: {leakage[:10]}")
    print("Shrimp-level leakage check passed.")
    remove_yolo_label_caches(base_path)


if GROUP_SPLIT_BY_SHRIMP:
    split_grouped_by_shrimp()


Building shrimp-grouped, disease-stratified split:
  - BG: 60 train groups, 7 valid groups, 8 test groups
  - Healthy: 113 train groups, 14 valid groups, 15 test groups
  - WSSV: 94 train groups, 11 valid groups, 13 test groups
  - WSSV_BG: 64 train groups, 8 valid groups, 9 test groups
Shrimp-grouped split complete:
  - train: 331 shrimp groups, 905 images
    group disease counts: {'BG': 60, 'Healthy': 113, 'WSSV': 94, 'WSSV_BG': 64}
    image disease counts: {'BG': 153, 'Healthy': 321, 'WSSV': 258, 'WSSV_BG': 173}
  - valid: 40 shrimp groups, 115 images
    group disease counts: {'BG': 7, 'Healthy': 14, 'WSSV': 11, 'WSSV_BG': 8}
    image disease counts: {'BG': 21, 'Healthy': 41, 'WSSV': 32, 'WSSV_BG': 21}
  - test: 45 shrimp groups, 129 images
    group disease counts: {'BG': 8, 'Healthy': 15, 'WSSV': 13, 'WSSV_BG': 9}
    image disease counts: {'BG': 24, 'Healthy': 41, 'WSSV': 38, 'WSSV_BG': 26}
Source filename disease counts before split: {'BG': 198, 'Healthy': 403, 'WSSV': 328, 

In [5]:
# Update data.yaml to absolute paths
import yaml
from pathlib import Path

base_path = "/kaggle/working/shrimpDisHandSegV2-1"
data_yaml_path = str(Path(base_path) / "data.yaml")

with open(data_yaml_path, "r") as f:
    content = yaml.safe_load(f)

content["train"] = str(Path(base_path) / "train" / "images")
content["val"] = str(Path(base_path) / "valid" / "images")
content["test"] = str(Path(base_path) / "test" / "images")

with open(data_yaml_path, "w") as f:
    yaml.safe_dump(content, f, sort_keys=False)

print("data.yaml updated:", data_yaml_path)
print(content)


data.yaml updated: /kaggle/working/shrimpDisHandSegV2-1/data.yaml
{'train': '/kaggle/working/shrimpDisHandSegV2-1/train/images', 'val': '/kaggle/working/shrimpDisHandSegV2-1/valid/images', 'test': '/kaggle/working/shrimpDisHandSegV2-1/test/images', 'nc': 2, 'names': ['BG', 'WSSV'], 'roboflow': {'workspace': 'lets-try-this', 'project': 'shrimpdishandsegv2', 'version': 1, 'license': 'CC BY 4.0', 'url': 'https://universe.roboflow.com/lets-try-this/shrimpdishandsegv2/dataset/1'}}


In [6]:
# Runtime patch architecture modules into Ultralytics namespace.
# This avoids fragile source-file text patches and keeps YAML parsing safe.
import torch
import torch.nn as nn

import ultralytics
import ultralytics.nn.modules as nn_modules
import ultralytics.nn.modules.conv as conv_module
import ultralytics.nn.tasks as tasks_module


class SimAM(nn.Module):
    def __init__(self, c1=None, e_lambda=1e-4):
        super().__init__()
        self.e_lambda = e_lambda
        self.activation = nn.Sigmoid()

    def forward(self, x):
        b, c, h, w = x.size()
        n = h * w - 1
        if n <= 0:
            return x
        x_mu = x - x.mean(dim=[2, 3], keepdim=True)
        denom = 4 * (x_mu.pow(2).sum(dim=[2, 3], keepdim=True) / n + self.e_lambda)
        y = x_mu.pow(2) / denom + 0.5
        return x * self.activation(y)


class CoordAtt(nn.Module):
    """Channel-preserving Coordinate Attention.

    Compatible with both clean parse_model calls (CoordAtt(c1, reduction))
    and older patched parse_model calls (CoordAtt(c1, c2, reduction)).
    """

    def __init__(self, c1, c2=None, reduction=32):
        super().__init__()
        if c2 is not None and c2 != c1 and reduction == 32:
            # Clean parse_model passes YAML args directly: [c1, reduction].
            reduction = c2
            c2 = c1
        c2 = c1 if c2 is None else c2
        mip = max(8, c1 // reduction)
        self.conv1 = nn.Conv2d(c1, mip, 1, 1, 0)
        self.bn1 = nn.BatchNorm2d(mip)
        self.act = nn.SiLU()
        self.conv_h = nn.Conv2d(mip, c2, 1, 1, 0)
        self.conv_w = nn.Conv2d(mip, c2, 1, 1, 0)
        self.proj = nn.Identity() if c1 == c2 else nn.Conv2d(c1, c2, 1, 1, 0)

    def forward(self, x):
        identity = self.proj(x)
        n, c, h, w = x.size()
        x_h = x.mean(dim=3, keepdim=True)
        x_w = x.mean(dim=2, keepdim=True).permute(0, 1, 3, 2)
        y = self.act(self.bn1(self.conv1(torch.cat([x_h, x_w], dim=2))))
        x_h, x_w = torch.split(y, [h, w], dim=2)
        x_w = x_w.permute(0, 1, 3, 2)
        return identity * self.conv_h(x_h).sigmoid() * self.conv_w(x_w).sigmoid()


class LargeKernelAttention(nn.Module):
    """Channel-preserving LKA. YAML args: [actual_channels]."""

    def __init__(self, c1, kernel_size=5, dilation_kernel_size=7, dilation=3):
        super().__init__()
        self.dw = nn.Conv2d(c1, c1, kernel_size, padding=kernel_size // 2, groups=c1)
        padding = dilation * (dilation_kernel_size - 1) // 2
        self.dw_d = nn.Conv2d(c1, c1, dilation_kernel_size, padding=padding, dilation=dilation, groups=c1)
        self.pw = nn.Conv2d(c1, c1, 1)
        self.gate = nn.Sigmoid()

    def forward(self, x):
        return x * self.gate(self.pw(self.dw_d(self.dw(x))))


for namespace in (conv_module, nn_modules, tasks_module):
    namespace.SimAM = SimAM
    namespace.CoordAtt = CoordAtt
    namespace.LargeKernelAttention = LargeKernelAttention

existing_all = list(getattr(nn_modules, "__all__", []))
for name in ["SimAM", "CoordAtt", "LargeKernelAttention"]:
    if name not in existing_all:
        existing_all.append(name)
nn_modules.__all__ = existing_all

print("Ultralytics:", ultralytics.__version__)
print("Registered modules:", SimAM.__name__, CoordAtt.__name__, LargeKernelAttention.__name__)


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics: 8.4.61
Registered modules: SimAM CoordAtt LargeKernelAttention


In [7]:
# Group D strategy config
EXPERIMENT_KEY = "label_smoothing_calibration"
EXPERIMENT_NAME = "YOLO11n-seg + SimAM+CA + label smoothing + confidence calibration"
STRATEGY_TAG = "D2_LabelSmoothing_Calibration"

LABEL_SMOOTHING = 0.05
CONFIDENCE_GRID = [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]
TRAIN_OVERRIDES = {
    "epochs": 100,
    "patience": 25,
}


In [8]:
# Write SimAM+CA architecture YAML and smoke-build the model.
# Important channel note for YOLO11n:
# P3/P4/P5 channels after width scaling are 64/128/256, not 256/512/1024.
from pathlib import Path

from ultralytics import YOLO

YAML_FILENAME = "yolo11n-seg-simam-ca-head-group-d.yaml"
YAML_CONTENT = r"""# YOLO11n-seg + CoordAtt->SimAM on P3/P4/P5 head outputs
nc: 2
scales:
  n: [0.50, 0.25, 1024]

backbone:
  - [-1, 1, Conv, [64, 3, 2]]
  - [-1, 1, Conv, [128, 3, 2]]
  - [-1, 2, C3k2, [256, False, 0.25]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [-1, 2, C3k2, [512, False, 0.25]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [-1, 2, C3k2, [512, True]]
  - [-1, 1, Conv, [1024, 3, 2]]
  - [-1, 2, C3k2, [1024, True]]
  - [-1, 1, SPPF, [1024, 5]]
  - [-1, 2, C2PSA, [1024]]

head:
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 6], 1, Concat, [1]]
  - [-1, 2, C3k2, [512, False]]

  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 4], 1, Concat, [1]]
  - [-1, 2, C3k2, [256, False]]

  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 13], 1, Concat, [1]]
  - [-1, 2, C3k2, [512, False]]

  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 10], 1, Concat, [1]]
  - [-1, 2, C3k2, [1024, True]]

  - [16, 1, CoordAtt, [64, 32]]
  - [23, 1, SimAM, []]
  - [19, 1, CoordAtt, [128, 32]]
  - [25, 1, SimAM, []]
  - [22, 1, CoordAtt, [256, 32]]
  - [27, 1, SimAM, []]
  - [[24, 26, 28], 1, Segment, [nc, 32, 256]]"""

EXPERIMENT_ROOT = Path("/kaggle/working/yolov11n_simam_NhomD") / EXPERIMENT_KEY
CFG_DIR = EXPERIMENT_ROOT / "cfg"
CFG_DIR.mkdir(parents=True, exist_ok=True)
yaml_path = CFG_DIR / YAML_FILENAME
yaml_path.write_text(YAML_CONTENT)
print("YAML written to:", yaml_path)

# Smoke build catches channel, concat-size, and custom-module registration errors before training.
smoke_model = YOLO(str(yaml_path))
smoke_model.info()
print("Smoke build OK:", EXPERIMENT_NAME)


YAML written to: /kaggle/working/yolov11n_simam_NhomD/label_smoothing_calibration/cfg/yolo11n-seg-simam-ca-head-group-d.yaml
YOLO11n-seg-simam-ca-head-group-d summary: 225 layers, 2,854,718 parameters, 2,854,702 gradients, 9.7 GFLOPs
Smoke build OK: YOLO11n-seg + SimAM+CA + label smoothing + confidence calibration


In [9]:
# Evaluation helpers reused from the clean baseline style
import gc
import shutil
import time
from pathlib import Path

import pandas as pd
import yaml
from ultralytics import YOLO

IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

COUNT_PENALTY_WEIGHT = 0.05
DISEASE_MISS_PENALTY_WEIGHT = 0.15
HEALTHY_FP_PENALTY_WEIGHT = 0.10
PREDICT_CONF_FOR_COUNT = 0.25

CLEAN_TRAIN_ARGS = {
    "auto_augment": None,
    "erasing": 0.0,
    "mosaic": 0.0,
    "mixup": 0.0,
    "cutmix": 0.0,
    "copy_paste": 0.0,
    "fliplr": 0.5,
    "flipud": 0.0,
    "hsv_h": 0.01,
    "hsv_s": 0.35,
    "hsv_v": 0.20,
    "degrees": 0.0,
    "translate": 0.05,
    "scale": 0.20,
    "shear": 0.0,
    "perspective": 0.0,
    "multi_scale": 0.0,
    "bgr": 0.0,
}


def disable_ultralytics_albumentations():
    try:
        import ultralytics.data.augment as yolo_augment
    except Exception as exc:
        print(f"Could not patch Ultralytics Albumentations hook: {exc}")
        return

    class NoOpAlbumentations:
        contains_spatial = False

        def __init__(self, *args, **kwargs):
            self.transform = None

        def __call__(self, labels):
            return labels

    yolo_augment.Albumentations = NoOpAlbumentations
    print("Ultralytics Albumentations hook disabled.")


def remove_yolo_label_caches(root):
    for cache_path in Path(root).glob("**/*.cache"):
        cache_path.unlink()


def find_image_for_label(image_dir, label_file):
    stem = Path(label_file).stem
    for ext in IMAGE_EXTENSIONS:
        candidate = Path(image_dir) / f"{stem}{ext}"
        if candidate.exists():
            return candidate
    return None


def write_data_yaml(dataset_dir, yaml_path, val_dir="valid", test_dir="test"):
    with open(data_yaml_path, "r") as f:
        content = yaml.safe_load(f)
    content["train"] = str(Path(dataset_dir) / "train" / "images")
    content["val"] = str(Path(dataset_dir) / val_dir / "images")
    content["test"] = str(Path(dataset_dir) / test_dir / "images")
    with open(yaml_path, "w") as f:
        yaml.safe_dump(content, f, sort_keys=False)
    return yaml_path


def copy_split_by_label_state(src_dataset, dst_dataset, split, want_labeled):
    src_images = Path(src_dataset) / split / "images"
    src_labels = Path(src_dataset) / split / "labels"
    dst_images = Path(dst_dataset) / split / "images"
    dst_labels = Path(dst_dataset) / split / "labels"
    dst_images.mkdir(parents=True, exist_ok=True)
    dst_labels.mkdir(parents=True, exist_ok=True)
    copied = 0
    for label_path in sorted(src_labels.glob("*.txt")):
        lines = [line.strip() for line in label_path.read_text().splitlines() if line.strip()]
        if bool(lines) != want_labeled:
            continue
        image_path = find_image_for_label(src_images, label_path.name)
        if image_path is None:
            continue
        shutil.copy2(image_path, dst_images / image_path.name)
        shutil.copy2(label_path, dst_labels / label_path.name)
        copied += 1
    return copied


def make_state_eval_dataset(src_dataset, exp_key, state_name, want_labeled):
    dst = EXPERIMENT_ROOT / f"dataset_{state_name}_eval"
    if dst.exists():
        shutil.rmtree(dst)
    for sub in ["images", "labels"]:
        (dst / "train" / sub).mkdir(parents=True, exist_ok=True)
    copied = {}
    for split in ["valid", "test"]:
        copied[split] = copy_split_by_label_state(src_dataset, dst, split, want_labeled=want_labeled)
    yaml_path = dst / f"data_{state_name}.yaml"
    write_data_yaml(dst, yaml_path)
    print(f"{state_name} eval dataset for {exp_key}: {copied}")
    return dst, yaml_path, copied


def metric_value(metrics, dotted_path, default=float("nan")):
    obj = metrics
    for part in dotted_path.split("."):
        if not hasattr(obj, part):
            return default
        obj = getattr(obj, part)
    try:
        return float(obj)
    except Exception:
        return default


def list_images(images_dir):
    image_paths = []
    for ext in IMAGE_EXTENSIONS:
        image_paths.extend(Path(images_dir).glob(f"*{ext}"))
    return sorted(image_paths)


def count_prediction_errors(model, images_dir, labels_dir, conf=PREDICT_CONF_FOR_COUNT):
    image_paths = list_images(images_dir)
    results = model.predict(source=[str(p) for p in image_paths], imgsz=640, conf=conf, verbose=False)
    box_errors, mask_errors, box_exact, mask_exact = [], [], [], []
    gt_total = pred_box_total = pred_mask_total = disease_images = 0
    disease_box_miss_images = disease_mask_miss_images = 0

    for image_path, result in zip(image_paths, results):
        label_path = Path(labels_dir) / f"{image_path.stem}.txt"
        gt_count = 0
        if label_path.exists():
            gt_count = len([line for line in label_path.read_text().splitlines() if line.strip()])
        box_count = len(result.boxes) if result.boxes is not None else 0
        mask_count = len(result.masks) if result.masks is not None else 0
        denom = max(1, gt_count)
        box_errors.append(abs(box_count - gt_count) / denom)
        mask_errors.append(abs(mask_count - gt_count) / denom)
        box_exact.append(float(box_count == gt_count))
        mask_exact.append(float(mask_count == gt_count))
        if gt_count > 0:
            disease_images += 1
            disease_box_miss_images += int(box_count == 0)
            disease_mask_miss_images += int(mask_count == 0)
        gt_total += gt_count
        pred_box_total += box_count
        pred_mask_total += mask_count

    return {
        "images": len(image_paths),
        "gt_total": gt_total,
        "pred_box_total": pred_box_total,
        "pred_mask_total": pred_mask_total,
        "box_count_mae": sum(box_errors) / len(box_errors) if box_errors else float("nan"),
        "mask_count_mae": sum(mask_errors) / len(mask_errors) if mask_errors else float("nan"),
        "box_count_exact": sum(box_exact) / len(box_exact) if box_exact else float("nan"),
        "mask_count_exact": sum(mask_exact) / len(mask_exact) if mask_exact else float("nan"),
        "disease_images": disease_images,
        "disease_box_miss_images": disease_box_miss_images,
        "disease_mask_miss_images": disease_mask_miss_images,
        "disease_box_miss_rate": disease_box_miss_images / disease_images if disease_images else float("nan"),
        "disease_mask_miss_rate": disease_mask_miss_images / disease_images if disease_images else float("nan"),
    }


def healthy_false_positive_summary(model, images_dir, conf=PREDICT_CONF_FOR_COUNT):
    image_paths = list_images(images_dir)
    if not image_paths:
        return {
            "healthy_images": 0,
            "healthy_mask_fp_rate": float("nan"),
            "healthy_box_fp_rate": float("nan"),
            "healthy_fp_masks_total": 0,
            "healthy_fp_masks_per_image": float("nan"),
            "healthy_avg_fp_confidence": float("nan"),
        }
    results = model.predict(source=[str(p) for p in image_paths], imgsz=640, conf=conf, verbose=False)
    images_with_box_fp = images_with_mask_fp = box_total = mask_total = 0
    confidences = []
    for result in results:
        box_count = len(result.boxes) if result.boxes is not None else 0
        mask_count = len(result.masks) if result.masks is not None else 0
        if box_count > 0:
            images_with_box_fp += 1
            try:
                confidences.extend([float(v) for v in result.boxes.conf.detach().cpu().tolist()])
            except Exception:
                pass
        if mask_count > 0:
            images_with_mask_fp += 1
        box_total += box_count
        mask_total += mask_count
    n = len(image_paths)
    return {
        "healthy_images": n,
        "healthy_mask_fp_rate": images_with_mask_fp / n,
        "healthy_box_fp_rate": images_with_box_fp / n,
        "healthy_fp_masks_total": mask_total,
        "healthy_fp_masks_per_image": mask_total / n,
        "healthy_avg_fp_confidence": sum(confidences) / len(confidences) if confidences else 0.0,
    }


def read_best_epoch_from_results(run_path):
    results_csv = Path(run_path) / "results.csv"
    if not results_csv.exists():
        return {}
    df = pd.read_csv(results_csv)
    df.columns = [c.strip() for c in df.columns]
    mask_col = "metrics/mAP50(M)"
    if mask_col not in df.columns:
        return {"epochs_ran": len(df)}
    best_idx = df[mask_col].idxmax()
    best = df.iloc[best_idx]
    last = df.iloc[-1]
    return {
        "epochs_ran": int(len(df)),
        "best_epoch_by_mask_map50": int(best["epoch"]) if "epoch" in df.columns else int(best_idx + 1),
        "best_val_mask_map50": float(best.get(mask_col, float("nan"))),
        "best_val_mask_map50_95": float(best.get("metrics/mAP50-95(M)", float("nan"))),
        "last_train_seg_loss": float(last.get("train/seg_loss", float("nan"))),
        "last_val_seg_loss": float(last.get("val/seg_loss", float("nan"))),
        "seg_loss_gap_val_minus_train": float(last.get("val/seg_loss", float("nan")) - last.get("train/seg_loss", float("nan"))),
    }


In [10]:
# Patch classification BCE with target smoothing.
# This Ultralytics build does not expose a label_smoothing train arg, so we patch
# v8DetectionLoss.__init__ and keep the original target shape/reduction.
import torch.nn.functional as F
import ultralytics.utils.loss as ult_loss


class SmoothedBCEWithLogitsLoss(torch.nn.Module):
    def __init__(self, smoothing=0.05):
        super().__init__()
        self.smoothing = float(smoothing)

    def forward(self, pred, target):
        if self.smoothing > 0:
            target = target * (1.0 - self.smoothing) + 0.5 * self.smoothing
        return F.binary_cross_entropy_with_logits(pred, target, reduction="none")


if not hasattr(ult_loss, "_group_d_original_v8_detection_loss_init"):
    ult_loss._group_d_original_v8_detection_loss_init = ult_loss.v8DetectionLoss.__init__


def _group_d_smoothed_v8_detection_loss_init(self, *args, **kwargs):
    ult_loss._group_d_original_v8_detection_loss_init(self, *args, **kwargs)
    self.bce = SmoothedBCEWithLogitsLoss(smoothing=LABEL_SMOOTHING)


ult_loss.v8DetectionLoss.__init__ = _group_d_smoothed_v8_detection_loss_init
print(f"Patched v8DetectionLoss BCE with label smoothing={LABEL_SMOOTHING}")


Patched v8DetectionLoss BCE with label smoothing=0.05


In [11]:
# Training
import json
from pathlib import Path

from ultralytics import YOLO

EXPERIMENT_ROOT = Path("/kaggle/working/yolov11n_simam_NhomD") / EXPERIMENT_KEY
RUNS_DIR = Path("/kaggle/working/runs/segment")
REPORT_DIR = EXPERIMENT_ROOT / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

dataset_dir = Path(base_path)
remove_yolo_label_caches(dataset_dir)
labeled_eval_dir, labeled_eval_yaml, _ = make_state_eval_dataset(dataset_dir, EXPERIMENT_KEY, "labeled_only", True)
healthy_eval_dir, healthy_eval_yaml, _ = make_state_eval_dataset(dataset_dir, EXPERIMENT_KEY, "healthy_only", False)

run_name = f"yolov11n_simam_{EXPERIMENT_KEY}"
disable_ultralytics_albumentations()

yolo = YOLO(str(yaml_path))
try:
    yolo.load("yolo11n-seg.pt")
    print("Loaded yolo11n-seg pretrained weights.")
except Exception as exc:
    print("Pretrained load warning:", exc)

train_args = {
    "data": str(data_yaml_path),
    "task": "segment",
    "imgsz": 640,
    "epochs": 100,
    "batch": 16,
    "patience": 30,
    "seed": 42,
    "deterministic": True,
    "workers": 0,
    "project": str(RUNS_DIR),
    "name": run_name,
    "exist_ok": True,
    "pretrained": True,
    "plots": True,
    "verbose": True,
}
train_args.update(CLEAN_TRAIN_ARGS)
train_args.update(TRAIN_OVERRIDES)

print("Group D strategy:", EXPERIMENT_NAME)
print("Training overrides:", json.dumps(TRAIN_OVERRIDES, indent=2, sort_keys=True))

start = time.time()
yolo.train(**train_args)
train_time_min = (time.time() - start) / 60

run_path = RUNS_DIR / run_name
best_model_path = run_path / "weights" / "best.pt"
EVAL_MODEL_PATH = best_model_path
print("Best checkpoint:", best_model_path)


labeled_only eval dataset for label_smoothing_calibration: {'valid': 74, 'test': 88}
healthy_only eval dataset for label_smoothing_calibration: {'valid': 41, 'test': 41}
Ultralytics Albumentations hook disabled.
Transferred 378/594 items from pretrained weights
Loaded yolo11n-seg pretrained weights.
Group D strategy: YOLO11n-seg + SimAM+CA + label smoothing + confidence calibration
Training overrides: {
  "epochs": 100,
  "patience": 25
}
New https://pypi.org/project/ultralytics/8.4.72 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.61 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=None, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/shrimpDisHandSegV2-1/data.yaml, degrees=0.0, deterministic=True, dev

In [12]:
# Confidence calibration on validation split.
# The mAP metrics are still YOLO val metrics; this sweep selects the predict() confidence
# used by count error and healthy false-positive summaries.
calibration_model = YOLO(str(best_model_path))
confidence_grid = globals().get("CONFIDENCE_GRID", [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50])
labeled_val = calibration_model.val(data=str(labeled_eval_yaml), split="val", imgsz=640, plots=False, verbose=False)
labeled_val_map50 = metric_value(labeled_val, "seg.map50")

calibration_rows = []
for conf in confidence_grid:
    labeled_count = count_prediction_errors(
        calibration_model,
        labeled_eval_dir / "valid" / "images",
        labeled_eval_dir / "valid" / "labels",
        conf=conf,
    )
    healthy_fp = healthy_false_positive_summary(
        calibration_model,
        healthy_eval_dir / "valid" / "images",
        conf=conf,
    )
    score = (
        labeled_val_map50
        - COUNT_PENALTY_WEIGHT * labeled_count["mask_count_mae"]
        - DISEASE_MISS_PENALTY_WEIGHT * labeled_count["disease_box_miss_rate"]
        - HEALTHY_FP_PENALTY_WEIGHT * healthy_fp["healthy_mask_fp_rate"]
    )
    calibration_rows.append(
        {
            "conf": conf,
            "labeled_val_mask_map50": labeled_val_map50,
            "labeled_val_pred_masks": labeled_count["pred_mask_total"],
            "labeled_val_mask_count_mae": labeled_count["mask_count_mae"],
            "labeled_val_disease_box_miss_rate": labeled_count["disease_box_miss_rate"],
            "healthy_val_mask_fp_rate": healthy_fp["healthy_mask_fp_rate"],
            "healthy_val_fp_masks_total": healthy_fp["healthy_fp_masks_total"],
            "healthy_aware_val_score": score,
        }
    )

calibration_df = pd.DataFrame(calibration_rows).sort_values("healthy_aware_val_score", ascending=False).reset_index(drop=True)
calibration_csv = REPORT_DIR / f"{EXPERIMENT_KEY}_confidence_calibration.csv"
calibration_df.to_csv(calibration_csv, index=False)
display(calibration_df)

CALIBRATED_CONF = float(calibration_df.iloc[0]["conf"])
EVAL_MODEL_PATH = best_model_path
print("Selected calibrated confidence:", CALIBRATED_CONF)
print("Saved calibration sweep:", calibration_csv)


Ultralytics 8.4.61 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO11n-seg-simam-ca-head-group-d summary (fused): 135 layers, 2,846,678 parameters, 0 gradients, 9.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2585.7±295.7 MB/s, size: 328.5 KB)
val: Scanning /kaggle/working/yolov11n_simam_NhomD/label_smoothing_calibration/dataset_labeled_only_eval/valid/labels... 74 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 74/74 1.0Kit/s 0.1s
val: New cache created: /kaggle/working/yolov11n_simam_NhomD/label_smoothing_calibration/dataset_labeled_only_eval/valid/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 2.3it/s 2.1s
                   all         74        102      0.629      0.449      0.463      0.183      0.556       0.35       0.33      0.113
Speed: 1.0ms preprocess, 7.5ms inference, 0.0ms loss, 12.1ms postprocess per image


,conf,labeled_val_mask_map50,labeled_val_pred_masks,labeled_val_mask_count_mae,labeled_val_disease_box_miss_rate,healthy_val_mask_fp_rate,healthy_val_fp_masks_total,healthy_aware_val_score
0,0.20,0.329551,60,0.448198,0.283784,0.146341,6,0.249940
1,0.15,0.329551,89,0.567568,0.216216,0.292683,12,0.239472
2,0.10,0.329551,156,0.943694,0.108108,0.317073,15,0.234443
3,0.25,0.329551,44,0.565315,0.445946,0.121951,5,0.222199
4,0.30,0.329551,36,0.648649,0.540541,0.121951,5,0.203843
5,0.40,0.329551,26,0.756757,0.662162,0.073171,3,0.185072
6,0.35,0.329551,27,0.750000,0.662162,0.097561,4,0.182971
7,0.45,0.329551,21,0.770270,0.716216,0.073171,3,0.176288
8,0.50,0.329551,17,0.817568,0.770270,0.073171,3,0.165815


Selected calibrated confidence: 0.2
Saved calibration sweep: /kaggle/working/yolov11n_simam_NhomD/label_smoothing_calibration/reports/label_smoothing_calibration_confidence_calibration.csv


In [13]:
# Evaluation: full test, labeled-only test, and healthy-negative false positives
eval_model_path = Path(globals().get("EVAL_MODEL_PATH", best_model_path))
eval_conf = float(globals().get("CALIBRATED_CONF", PREDICT_CONF_FOR_COUNT))
best_model = YOLO(str(eval_model_path))

full_test = best_model.val(data=str(data_yaml_path), split="test", imgsz=640, plots=True, verbose=False)
labeled_test = best_model.val(data=str(labeled_eval_yaml), split="test", imgsz=640, plots=False, verbose=False)

test_count = count_prediction_errors(
    best_model,
    Path(base_path) / "test" / "images",
    Path(base_path) / "test" / "labels",
    conf=eval_conf,
)
labeled_test_count = count_prediction_errors(
    best_model,
    labeled_eval_dir / "test" / "images",
    labeled_eval_dir / "test" / "labels",
    conf=eval_conf,
)
healthy_test_fp = healthy_false_positive_summary(best_model, healthy_eval_dir / "test" / "images", conf=eval_conf)

labeled_map50 = metric_value(labeled_test, "seg.map50")
healthy_aware_score = (
    labeled_map50
    - COUNT_PENALTY_WEIGHT * labeled_test_count["mask_count_mae"]
    - DISEASE_MISS_PENALTY_WEIGHT * labeled_test_count["disease_box_miss_rate"]
    - HEALTHY_FP_PENALTY_WEIGHT * healthy_test_fp["healthy_mask_fp_rate"]
)

row = {
    "experiment": EXPERIMENT_KEY,
    "name": EXPERIMENT_NAME,
    "strategy": STRATEGY_TAG,
    "training_overrides": json.dumps(TRAIN_OVERRIDES, sort_keys=True),
    "run_name": run_name,
    "run_path": str(run_path),
    "best_pt": str(best_model_path),
    "eval_pt": str(eval_model_path),
    "eval_predict_conf": eval_conf,
    "train_time_min": round(train_time_min, 2),
    "full_test_box_map50": metric_value(full_test, "box.map50"),
    "full_test_mask_map50": metric_value(full_test, "seg.map50"),
    "full_test_mask_map50_95": metric_value(full_test, "seg.map"),
    "labeled_test_box_map50": metric_value(labeled_test, "box.map50"),
    "labeled_test_mask_map50": labeled_map50,
    "labeled_test_mask_map50_95": metric_value(labeled_test, "seg.map"),
    "test_gt_instances": test_count["gt_total"],
    "test_pred_masks": test_count["pred_mask_total"],
    "test_mask_count_mae": test_count["mask_count_mae"],
    "labeled_test_gt_instances": labeled_test_count["gt_total"],
    "labeled_test_pred_masks": labeled_test_count["pred_mask_total"],
    "labeled_test_mask_count_mae": labeled_test_count["mask_count_mae"],
    "labeled_test_disease_box_miss_rate": labeled_test_count["disease_box_miss_rate"],
    "labeled_test_disease_mask_miss_rate": labeled_test_count["disease_mask_miss_rate"],
    "healthy_test_images": healthy_test_fp["healthy_images"],
    "healthy_test_mask_fp_rate": healthy_test_fp["healthy_mask_fp_rate"],
    "healthy_test_box_fp_rate": healthy_test_fp["healthy_box_fp_rate"],
    "healthy_test_fp_masks_total": healthy_test_fp["healthy_fp_masks_total"],
    "healthy_test_fp_masks_per_image": healthy_test_fp["healthy_fp_masks_per_image"],
    "healthy_aware_labeled_test_mask_map50": healthy_aware_score,
}
row.update(read_best_epoch_from_results(run_path))

summary_df = pd.DataFrame([row])
summary_csv = REPORT_DIR / f"{EXPERIMENT_KEY}_summary.csv"
summary_df.to_csv(summary_csv, index=False)
display(summary_df)
print("Saved summary:", summary_csv)

del yolo, best_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


Ultralytics 8.4.61 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO11n-seg-simam-ca-head-group-d summary (fused): 135 layers, 2,846,678 parameters, 0 gradients, 9.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2290.6±244.1 MB/s, size: 304.8 KB)
val: Scanning /kaggle/working/shrimpDisHandSegV2-1/test/labels... 129 images, 41 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 129/129 1.2Kit/s 0.1s
val: New cache created: /kaggle/working/shrimpDisHandSegV2-1/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 1.6it/s 5.6s
                   all        129        119      0.312      0.397      0.233     0.0819      0.271      0.349      0.201      0.048
Speed: 3.7ms preprocess, 6.4ms inference, 0.0ms loss, 13.5ms postprocess per image
Results saved to /kaggle/working/runs/segment/val-2
Ultralytics 8.4.61 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 

,experiment,name,strategy,training_overrides,run_name,run_path,best_pt,eval_pt,eval_predict_conf,train_time_min,...,healthy_test_fp_masks_total,healthy_test_fp_masks_per_image,healthy_aware_labeled_test_mask_map50,epochs_ran,best_epoch_by_mask_map50,best_val_mask_map50,best_val_mask_map50_95,last_train_seg_loss,last_val_seg_loss,seg_loss_gap_val_minus_train
0,label_smoothing_calibration,YOLO11n-seg + SimAM+CA + label smoothing + con...,D2_LabelSmoothing_Calibration,"{""epochs"": 100, ""patience"": 25}",yolov11n_simam_label_smoothing_calibration,/kaggle/working/runs/segment/yolov11n_simam_la...,/kaggle/working/runs/segment/yolov11n_simam_la...,/kaggle/working/runs/segment/yolov11n_simam_la...,0.2,74.8,...,59,1.439024,0.184269,100,100,0.29192,0.09619,3.06772,3.61785,0.55013


Saved summary: /kaggle/working/yolov11n_simam_NhomD/label_smoothing_calibration/reports/label_smoothing_calibration_summary.csv
